In [48]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\teste"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico NextPower - Tags.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico NextPower - Tags.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico NextPower - Tags.dwg")

conn = load_to_duckdb(dxf_file, "PV Paiva")
print("Data Loaded")
print("DuckDB Loaded")


os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Paiva already exists
Removed existing data for project: PV Paiva
Successfully loaded project
Loaded 12852 entities for project: PV Paiva
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [49]:
data = conn.sql("SELECT type, layer, name, color, X, Y, UFV, PST, INV, STR, GRP FROM cad_entities WHERE project_name = 'PV Paiva'")
print(data)


┌─────────┬────────────────────┬──────────────────────────────────┬─────────┬────────────┬────────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│  type   │       layer        │               name               │  color  │     X      │     Y      │   UFV   │   PST   │   INV   │   STR   │   GRP   │
│ varchar │      varchar       │             varchar              │ varchar │   float    │   float    │ varchar │ varchar │ varchar │ varchar │ varchar │
├─────────┼────────────────────┼──────────────────────────────────┼─────────┼────────────┼────────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ INSERT  │ 0                  │ EMF397_PST                       │ 256     │  34592.234 │   127259.4 │         │ 22      │ NULL    │ NULL    │ NULL    │
│ INSERT  │ 0                  │ EMF397_PST                       │ 256     │ 34995.4375 │   127259.4 │         │ 23      │ NULL    │ NULL    │ NULL    │
│ INSERT  │ 0                  │ EMF397_PST                       │ 256     

In [50]:
result = conn.sql("""
    SELECT
        layer,
        COUNT(*) as count
    FROM data
    GROUP BY layer
    ORDER BY count
""")
print(result)

┌────────────────────┬───────┐
│       layer        │ count │
│      varchar       │ int64 │
├────────────────────┼───────┤
│ EMF_em_pst_26      │    14 │
│ EMF_em_pst_25      │    15 │
│ EMF_em_pst_11      │    16 │
│ EMF_em_pst_21      │    16 │
│ EMF_em_pst_29      │    16 │
│ EMF_em_pst_03      │    16 │
│ EMF_em_pst_08      │    16 │
│ EMF_em_pst_27      │    16 │
│ EMF_em_pst_14      │    16 │
│ EMF_em_pst_17      │    16 │
│       ·            │     · │
│       ·            │     · │
│       ·            │     · │
│ EMF_em_pst_19      │    16 │
│ EMF_em_pst_15      │    16 │
│ 0                  │    28 │
│ EMF_em_pst_13      │   120 │
│ EMF_em_pst_04      │   150 │
│ EMF_em_pst_09      │   442 │
│ EMF_em_pst_07      │   442 │
│ EMF_em_tracker_32p │   948 │
│ EMF_em_tracker_64p │  2672 │
│ EMF_em_tracker_96p │  7653 │
└────────────────────┴───────┘
  33 rows          2 columns
  (20 shown)                 



In [51]:
inverters = conn.sql("SELECT * FROM data WHERE name LIKE '%inversor%'")
inverters_df = inverters.fetchdf()
inverters_df['UFV'] = 0
inverters_df['PST'] = inverters_df['layer'].astype(str).str[-2:]
print(inverters_df)

       type          layer             name color             X  \
0    INSERT  EMF_em_pst_29  EMF397_inversor   256  34286.687500   
1    INSERT  EMF_em_pst_29  EMF397_inversor   256  34368.585938   
2    INSERT  EMF_em_pst_29  EMF397_inversor   256  34463.085938   
3    INSERT  EMF_em_pst_29  EMF397_inversor   256  34494.585938   
4    INSERT  EMF_em_pst_29  EMF397_inversor   256  34538.687500   
..      ...            ...              ...   ...           ...   
457  INSERT  EMF_em_pst_15  EMF397_inversor   256  34368.585938   
458  INSERT  EMF_em_pst_15  EMF397_inversor   256  34425.285156   
459  INSERT  EMF_em_pst_09  EMF397_inversor     9  34291.253906   
460  INSERT  EMF_em_pst_04  EMF397_inversor     4  34335.351562   
461  INSERT  EMF_em_pst_09  EMF397_inversor     4  34272.355469   

                 Y  UFV PST INV STR   GRP  
0    125996.968750    0  29  15      None  
1    125996.968750    0  29  16      None  
2    126045.968750    0  29   9      None  
3    126045.968750 

In [56]:
inverters_df['GRP'] = inverters_df["UFV"].astype(str) + "." + inverters_df["PST"].astype(str)
inverters_df.sort_values(by=['GRP', 'Y', 'X'], ascending=[True, False, True], inplace=True, ignore_index=True)
inverters_df["INV"] = (inverters_df.groupby("GRP").cumcount()+1).astype(str).str.zfill(2)
inverters_df['grp.color'] = inverters_df['GRP'].astype(str) + "." + inverters_df['color'].astype(str)

print(inverters_df)

       type          layer             name color             X  \
0    INSERT  EMF_em_pst_01  EMF397_inversor   256  36054.707031   
1    INSERT  EMF_em_pst_01  EMF397_inversor   256  35947.609375   
2    INSERT  EMF_em_pst_01  EMF397_inversor   256  35827.906250   
3    INSERT  EMF_em_pst_01  EMF397_inversor   256  35752.304688   
4    INSERT  EMF_em_pst_01  EMF397_inversor   256  35443.605469   
..      ...            ...              ...   ...           ...   
457  INSERT  EMF_em_pst_29  EMF397_inversor   256  34589.085938   
458  INSERT  EMF_em_pst_29  EMF397_inversor   256  34645.785156   
459  INSERT  EMF_em_pst_29  EMF397_inversor   256  34544.988281   
460  INSERT  EMF_em_pst_29  EMF397_inversor   256  34286.687500   
461  INSERT  EMF_em_pst_29  EMF397_inversor   256  34368.585938   

                 Y  UFV PST INV STR   GRP grp+color grp.color  
0    129708.257812    0  01  01      0.01  0.01.256  0.01.256  
1    129703.062500    0  01  02      0.01  0.01.256  0.01.256  
2  

In [53]:
strings = conn.sql("SELECT * FROM data WHERE name LIKE '%Tracker%'")
strings_df = strings.fetchdf()
strings_df['UFV'] = 0
strings_df['PST'] = strings_df['layer'].astype(str).str[-2:]
print(strings_df)

         type               layer                              name color  \
0      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
1      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
2      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
3      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
4      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
...       ...                 ...                               ...   ...   
12357  INSERT       EMF_em_pst_07  String_Tracker_1x96p_Ponta_North   256   
12358  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
12359  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
12360  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
12361  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   

                  X              Y  UFV PST INV STR   GRP  
0      34421.10

In [61]:
strings9 = strings_df[strings_df['layer'] == 'EMF_em_pst_09']
strings9['grp.color'] = strings9['UFV'].astype(str) + "." + strings9['PST'].astype(str) + '.' + strings9['color'].astype(str)
strings9.drop(columns=['INV'], inplace=True)
print(strings9)

         type          layer                        name color             X  \
67     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16  34070.753906   
68     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North     5  34184.152344   
69     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16  34039.253906   
70     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16  34045.554688   
71     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16  34051.851562   
...       ...            ...                         ...   ...           ...   
12319  INSERT  EMF_em_pst_09  String_Tracker_1x64p_North     9  34316.449219   
12320  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     8  34240.851562   
12321  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     8  34247.148438   
12322  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     8  34253.449219   
12346  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North    16  34026.648438   

                   Y  UFV PST STR   GRP

In [62]:
emf9 = strings9.merge(inverters_df, on='grp.color', how='left')
print(emf9)

     type_x        layer_x                      name_x color_x           X_x  \
0    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North      16  34070.753906   
1    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North       5  34184.152344   
2    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North      16  34039.253906   
3    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North      16  34045.554688   
4    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North      16  34051.851562   
..      ...            ...                         ...     ...           ...   
475  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North       8  34247.148438   
476  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North       8  34247.148438   
477  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North       8  34253.449219   
478  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North       8  34253.449219   
479  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North      16  34026.648438   

               Y_x  UFV_x PST_x STR_x G